In [17]:
import os
import sys
import torch
import tiktoken

project_root = os.path.dirname(os.path.abspath("")) # since notebook is in evaluation/
sys.path.insert(0, project_root)

import model
from model.model import GPT, GPTConfig
model.GPTConfig = GPTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"Using device: {device}")

Using device: mps


In [18]:
# Load configuration matching the notebook setup
config = GPTConfig(
    block_size=256,
    vocab_size=50257,
    n_layer=8,
    n_head=8,
    n_embd=384,
    dropout=0.1
)

# Initialize model
model = GPT(config)
model.to(device)
model.eval()

model_path = os.path.join(project_root, "training", "nanogpt_checkpoint_8.pt")
if not os.path.exists(model_path):
    print(f"Error: {model_path} not found.")
    print("Please run the notebook 'training/training_pipeline.ipynb' to train and save the model.")
else:
    # Load weights
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if "model" in checkpoint:
        model.load_state_dict(checkpoint["model"])
    else:
        model.load_state_dict(checkpoint)
    print(f"Loaded model from {model_path}")

number of parameters: 33.50M
Loaded model from /Users/idant/Developer/Projects/NanoGPT/training/nanogpt_checkpoint_8.pt


In [19]:
# Setup tokenizer
enc = tiktoken.get_encoding("gpt2")

prompt = "[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]\nYeah, look, it's 3 AM in Toronto"
print(f"\nPrompt: '{prompt}'\n")

idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
max_new_tokens = 500


Prompt: '[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]
Yeah, look, it's 3 AM in Toronto'



### Evaluation

In [29]:
print("Greedy Decoding")
out_idx = model.generate(idx, max_new_tokens, top_k=1, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

Greedy Decoding
[Genre: boom_bap] [Mood: aggressive] [Rhyme: dense_internal] [Cadence: fast]
Look, his palms are sweaty, knees weak, arms are heavy
And I'm a savage, my diamonds is falling, hands is falling (Yeah)
I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'm a savage, I'

In [21]:
print("Temperature Sampling")
torch.manual_seed(42)
out_idx = model.generate(idx, 100, temperature=0.7, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

Temperature Sampling
[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]
Yeah, look, it's 3 AM in Toronto
You know that nigga let me see you my style and you know I'm coming back to your socks (Woo)
Bitch, take a new toy-up nigga-ass nigga, get the fuck outta here
I ain't got no mind or nothin' but no stoppin'
It's a big bitch, this is a big bitches can't reach (Haha)
We gon' make love for none of these niggas that we


In [22]:
print("Top-k Sampling")
torch.manual_seed(42)
out_idx = model.generate(idx, 100, top_k=15, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

Top-k Sampling
[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]
Yeah, look, it's 3 AM in Toronto
You see me and you know I'm a star' of shit
You see me when ya come to the moon on this bitch<|endoftext|>


In [23]:
print("Top-p Sampling")
torch.manual_seed(42)
out_idx = model.generate(idx, 100, top_p=0.8, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

Top-p Sampling
[Genre: melodic_rap] [Mood: vulnerable] [Rhyme: internal_rhyme] [Cadence: midtempo]
Yeah, look, it's 3 AM in Toronto
Might need to send you a check on my style and bring me back
You can show your body like my name is (Bitch)
Holla when we rollin' at your hands, bitch

Ayy, woah
Yo, you know what I'm saying?
Go ahead of reality, fuck the world
And the real people come take down my mind again
For the wrong have changed from killing, I got ninety-nine
Been like some free


### Testing for hallucinations

In [25]:
prompt = "[Artist: MC Fictional][Song: Fictional]\n This is a fictional song"
idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
out_idx = model.generate(idx, 200, temperature=0.7, repetition_penalty=1.2, eos_id=enc.eot_token)
print(enc.decode(out_idx[0].tolist()))

[Artist: MC Fictional][Song: Fictional]
 This is a fictional song for me to see us by the day today, or a lot of course?
Maybe it's hard enough but it's freedom for us
And we've been programmed, and the same ones who hate us
It takes all our own love for us, for us, like, our kids and they're listenin' on (Ugh)
You don't give an example, 'cause you cannot get arrested (Ayy)
This isn't no more than them in your chest is nothin' new
For all these waters do is either have given to see us with us
But it'd be a time to takeication from our minds without himself
Oh, oh, yeah<|endoftext|>
